# Lightweight Testing
**Objective:** Practice translating small scientific checks into executable tests that keep angle arithmetic honest.


Unit testing is a pragmatic way to document expectations, surface bugs quickly, and build confidence before a notebook graduates into a production pipeline. 
We will keep the iterative, bug-fix narrative from the earlier section and show how the same checks look in both `unittest` and `pytest` so that you can choose the style that fits your workflow.


## Example: Calculate the distance between two angles (in degrees)
We will prototype a helper that returns the smallest distance between two bearings. The first version is intentionally naive so that each failing test points us toward a sturdier implementation.


In [ ]:
def angle_distance(a, b):
    """Return the absolute difference between two angles in degrees."""
    return abs(b - a)


### Writing our first test
We'll begin with one happy-path test that ensures the distance between two "small" angles behaves the way a human calculator would expect.


In [ ]:
import unittest


class TestAngle(unittest.TestCase):
    def test_small_angles(self):
        """Test distance between small angles."""
        a = 10
        b = 90
        expected = 80
        result = angle_distance(a, b)
        self.assertEqual(result, expected)


### Running the test
Calling `unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle')` confines execution to the test class we just defined and keeps Jupyter from interpreting extra CLI arguments.


In [ ]:
# Using defaultTest ensures we only run the freshly defined TestAngle class in this cell.
unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle', verbosity=2);


### Add test for edge case 1
We should also cover an example where the naive implementation fails: the gap between 0° and 270° should be 90°, not 270°.


In [ ]:
class TestAngle(unittest.TestCase):
    def test_small_angles(self):
        a = 10
        b = 90
        expected = 80
        self.assertEqual(angle_distance(a, b), expected)

    def test_large_angles(self):
        a = 0
        b = 270
        expected = 90
        self.assertEqual(angle_distance(a, b), expected)


In [ ]:
unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle', verbosity=2);


### Fix the first bug
Those failures show that we need to wrap the difference once it exceeds 180°. A quick fix is to use modulo arithmetic so that anything bigger than a semicircle folds back on itself.


In [ ]:
def angle_distance(a, b):
    """Return angle distance, folding differences larger than 180 degrees."""
    return abs(b - a) % 180


In [ ]:
unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle', verbosity=2);


### Add test for edge case 2
What about bearings that straddle the 0°/360° boundary? The shortest distance between 1° and 359° is only 2°, so we add a test that proves whether our fix handled wrap-around correctly.


In [ ]:
class TestAngle(unittest.TestCase):
    def test_small_angles(self):
        self.assertEqual(angle_distance(10, 90), 80)

    def test_large_angles(self):
        self.assertEqual(angle_distance(0, 270), 90)

    def test_wrapping_angles(self):
        self.assertEqual(angle_distance(1, 359), 2)


In [ ]:
unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle', verbosity=2);


### Fix the second bug
The modulo trick was not enough because it mishandles differences that fall between 180° and 360°. Instead we compute the straight difference and explicitly choose the shorter arc.


In [ ]:
def angle_distance(a, b):
    d = abs(b - a)
    return min(360 - d, d)


In [ ]:
unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle', verbosity=2);


### Add test for edge case 3
Angles might already be expressed beyond 360° (e.g., the 720° output of a motion model). Let's add a test to make sure we normalize both inputs.


In [ ]:
class TestAngle(unittest.TestCase):
    def test_small_angles(self):
        self.assertEqual(angle_distance(10, 90), 80)

    def test_large_angles(self):
        self.assertEqual(angle_distance(0, 270), 90)

    def test_wrapping_angles(self):
        self.assertEqual(angle_distance(1, 359), 2)

    def test_large_input_angles(self):
        self.assertEqual(angle_distance(720, 270), 90)


In [ ]:
unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle', verbosity=2);


### Fix the third bug
Normalizing the inputs to the `[0, 360)` range before measuring the distance finishes the implementation.


In [ ]:
def angle_distance(a, b):
    d = abs((b % 360) - (a % 360))
    return min(360 - d, d)


In [ ]:
unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle', verbosity=2);


### Cleaning up the tests with `subTest`
These assertions differ only by parameter values, so we can tighten the code and get clearer failure messages with `subTest`.


In [ ]:
class TestAngle(unittest.TestCase):
    def test_angle_distance_cases(self):
        a_values = [10, 0, 1, 720]
        b_values = [90, 270, 359, 270]
        expected_values = [80, 90, 2, 90]
        for a, b, expected in zip(a_values, b_values, expected_values):
            with self.subTest(a=a, b=b):
                self.assertEqual(angle_distance(a, b), expected)


In [ ]:
unittest.main(argv=['first-arg-is-ignored'], exit=False, defaultTest='TestAngle', verbosity=2);


If any of the sub-tests fail, the error message lists the `a` and `b` values that triggered the regression.


### Using coverage to see what we missed
The `coverage` command-line tool highlights which lines executed during the suite. Run the following from the repository root (or wherever your tests live) to verify that every branch in `angle_distance` is exercised.


In [ ]:
# Measure coverage for the unittest version
# coverage run -m unittest discover -s classes/class1 -p 'test_*.py'
# coverage report -m
#
# Example console output
# Name                                            Stmts   Miss  Cover
# -------------------------------------------------------------------
# classes/class1/angle_demo.py      8      0   100%
# tests/test_angle_distance.py                      24      0   100%
# -------------------------------------------------------------------
# TOTAL                                             32      0   100%
#
# Focus on the `Miss` column—any non-zero counts tell you where to add new tests.


### Pytest equivalent
`pytest` encourages small test functions plus parameterization. The logic below mirrors the exact expectations from the `unittest` suite.


In [ ]:
import pytest


@pytest.mark.parametrize(
    ("angle_a", "angle_b", "expected"),
    [
        (10, 90, 80),
        (0, 270, 90),
        (1, 359, 2),
        (720, 270, 90),
    ],
    ids=["small angles", "three-quarter turn", "wrap across zero", "inputs above 360"],
)
def test_angle_distance_pytest(angle_a, angle_b, expected):
    assert angle_distance(angle_a, angle_b) == expected


# Drive the parametrized function manually so this cell proves it matches the unittest expectations.
for mark in getattr(test_angle_distance_pytest, 'pytestmark', []):
    if mark.name == 'parametrize':
        for angle_a, angle_b, expected in mark.args[1]:
            test_angle_distance_pytest(angle_a, angle_b, expected)
            print(f"[pass] angle_a={angle_a}, angle_b={angle_b} -> {expected}")

print("Pytest-style parametrized assertions passed for every case. Run `pytest -q` to see the official report.")


## Things to keep in mind about writing tests
- Only write tests for code you own—treat upstream libraries as contracts, not targets.
- Every function should have at least one test, and every `if/else` branch should get its own scenario (coverage helps you verify this).
- Whenever you fix a bug, first add a failing test that proves the bug exists.
- Pick either `unittest` or `pytest` for day-to-day work, but understand both so you can collaborate across teams.
